# 5장. 서식 있는 엑셀 보고서로 내보내기

받는 사람은 엑셀로 연다. 숫자에 쉼표가 있고, 열 너비가 맞고, 그래프가 붙은 파일을 만든다.

In [1]:
from pathlib import Path

import pandas as pd
from openpyxl.chart import BarChart, Reference
from openpyxl.formatting.rule import CellIsRule
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

WORK = Path("../work")
OUTPUT = Path("../output")
WEEK = "2026-11-02"

summary = pd.read_csv(WORK / "04_summary.csv")
pivot = pd.read_csv(WORK / "04_pivot.csv", index_col="지점")
top_items = pd.read_csv(WORK / "04_top_items.csv")
sales = pd.read_csv(WORK / "03_clean.csv", parse_dates=["판매일"])
check = pd.read_csv(WORK / "03_check.csv", parse_dates=["판매일"])

## 1. 일단 그냥 저장해 보기
`to_excel` 한 줄이면 파일은 나온다. 열어 보면 쉼표도 없고 열이 좁아 보고서로 보내기는 어렵다.

In [2]:
OUTPUT.mkdir(exist_ok=True)
summary.to_excel(OUTPUT / "연습_서식없음.xlsx", index=False)
print("저장:", "output/연습_서식없음.xlsx")

저장: output/연습_서식없음.xlsx


## 2. 표 하나를 꾸미는 함수
머리글은 굵게 하고 옅은 회색을 칠한다. 열마다 숫자 서식과 너비를 받는다.
`"#,##0"`과 `"0.0%"`는 엑셀 셀 서식 창의 '사용자 지정'에 적는 것과 같은 글자다.

In [3]:
HEADER_FILL = PatternFill("solid", fgColor="E7E6E6")
MONEY = "#,##0"
PERCENT = "0.0%"
DATE = "yyyy-mm-dd"


def style_table(ws, header_row, formats=None, widths=None):
    """머리글을 굵게 칠하고, 열마다 숫자 서식과 너비를 준다."""
    for cell in ws[header_row]:
        if cell.value is not None:
            cell.font = Font(bold=True)
            cell.fill = HEADER_FILL
            cell.alignment = Alignment(horizontal="center")
    for col, number_format in (formats or {}).items():
        for cell in ws[col][header_row:]:
            cell.number_format = number_format
    for col, width in (widths or {}).items():
        ws.column_dimensions[col].width = width

## 3. 시트 네 장짜리 보고서
요약(지점별 표, 많이 팔린 품목, 그래프), 품목군별, 상세, 확인필요.

In [4]:
def write_report(path, week, summary, pivot, top_items, sales, check):
    """표 다섯 개를 시트 네 장짜리 엑셀 파일로 저장한다."""
    monday = pd.Timestamp(week)
    sunday = monday + pd.Timedelta(days=6)
    n = len(summary)

    total = pd.DataFrame([{
        "지점": "전체",
        "금액": summary["금액"].sum(),
        "주간목표": summary["주간목표"].sum(),
        "달성률": summary["금액"].sum() / summary["주간목표"].sum(),
        "전주금액": summary["전주금액"].sum(),
        "전주대비": summary["금액"].sum() / summary["전주금액"].sum() - 1,
    }])
    summary_out = pd.concat([summary, total], ignore_index=True)
    top_row = n + 7  # 요약 표 아래 두 줄 띄우고 '많이 팔린 품목' 제목

    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        summary_out.to_excel(writer, sheet_name="요약", index=False, startrow=2)
        top_items.to_excel(writer, sheet_name="요약", index=False, startrow=top_row)
        pivot.to_excel(writer, sheet_name="품목군별")
        sales.to_excel(writer, sheet_name="상세", index=False)
        check.to_excel(writer, sheet_name="확인필요", index=False)

        # 요약 시트
        ws = writer.sheets["요약"]
        ws["A1"] = f"주간 판매 보고서 ({monday:%m/%d}~{sunday:%m/%d})"
        ws["A1"].font = Font(bold=True, size=14)
        top_title = ws.cell(row=top_row, column=1, value="많이 팔린 품목 5개")
        top_title.font = Font(bold=True)
        style_table(
            ws, header_row=3,
            formats={"B": MONEY, "C": MONEY, "D": PERCENT, "E": MONEY, "F": PERCENT},
            widths={"A": 14, "B": 18, "C": 14, "D": 12, "E": 14, "F": 12},
        )
        style_table(ws, header_row=top_row + 1)
        for row in ws.iter_rows(
            min_row=top_row + 2, max_row=top_row + 1 + len(top_items),
            min_col=3, max_col=4,
        ):
            for cell in row:
                cell.number_format = MONEY
        for cell in ws[3 + n + 1]:  # '전체' 줄
            cell.font = Font(bold=True)
        # 목표에 못 미친 달성률은 빨간 글자
        ws.conditional_formatting.add(
            f"D4:D{3 + n}",
            CellIsRule(
                operator="lessThan", formula=["1"],
                font=Font(color="C00000", bold=True),
            ),
        )
        # 지점별 금액과 목표 막대 그래프
        chart = BarChart()
        chart.title = "지점별 판매 금액과 목표"
        chart.title.overlay = False  # 제목과 범례가 막대 위에 겹치지 않게
        chart.y_axis.numFmt = MONEY
        chart.x_axis.delete = False  # 이 두 줄이 없으면 축 글자가 보이지 않는다
        chart.y_axis.delete = False
        chart.legend.position = "b"
        chart.legend.overlay = False
        chart.add_data(
            Reference(ws, min_col=2, max_col=3, min_row=3, max_row=3 + n),
            titles_from_data=True,
        )
        chart.set_categories(Reference(ws, min_col=1, min_row=4, max_row=3 + n))
        chart.width, chart.height = 16, 8.5
        ws.add_chart(chart, "H3")
        ws.page_setup.orientation = "landscape"
        ws.page_setup.fitToWidth = 1
        ws.page_setup.fitToHeight = 0
        ws.sheet_properties.pageSetUpPr.fitToPage = True

        # 품목군별 시트
        ws = writer.sheets["품목군별"]
        cols = [get_column_letter(i) for i in range(2, ws.max_column + 1)]
        style_table(
            ws, 1,
            formats={c: MONEY for c in cols},
            widths={"A": 12, **{c: 14 for c in cols}},
        )
        for cell in ws[ws.max_row]:
            cell.font = Font(bold=True)

        # 상세, 확인필요 시트
        widths = {
            "A": 9, "B": 22, "C": 15, "D": 12, "E": 10,
            "F": 20, "G": 8, "H": 10, "I": 12, "J": 26,
        }
        for name in ["상세", "확인필요"]:
            ws = writer.sheets[name]
            style_table(
                ws, 1,
                formats={"D": DATE, "G": MONEY, "H": MONEY, "I": MONEY},
                widths=widths,
            )
            ws.freeze_panes = "A2"
        writer.sheets["상세"].auto_filter.ref = writer.sheets["상세"].dimensions


report_path = OUTPUT / f"주간판매보고서_{WEEK}.xlsx"
write_report(report_path, WEEK, summary, pivot, top_items, sales, check)
print("저장:", f"output/{report_path.name}")

저장: output/주간판매보고서_2026-11-02.xlsx


## 4. 저장한 파일을 다시 읽어 검산
파일을 보내기 전에, 저장된 파일에서 합계를 다시 읽어 계산한 값과 견준다.

In [5]:
saved = pd.read_excel(report_path, sheet_name=None)  # 모든 시트를 한 번에
for name, table in saved.items():
    print(f"{name}: {table.shape[0]}줄 × {table.shape[1]}열")

same = int(saved["상세"]["금액"].sum()) == int(sales["금액"].sum())
print(
    f"상세 시트 합계 {int(saved['상세']['금액'].sum()):,}원 →",
    "계산한 값과 같음" if same else "다름!",
)

요약: 17줄 × 6열
품목군별: 6줄 × 6열
상세: 253줄 × 9열
확인필요: 1줄 × 10열
상세 시트 합계 29,919,700원 → 계산한 값과 같음


## 실습. 달성률 순으로 줄 세운 시트 하나 더 만들기
이미 만든 파일에 시트를 덧붙일 때는 `mode="a"`로 연다.

In [6]:
ranked = summary.sort_values("달성률", ascending=False)
ranked = ranked[["지점", "금액", "주간목표", "달성률"]]

practice_path = OUTPUT / "연습_시트추가.xlsx"
summary.to_excel(practice_path, sheet_name="요약", index=False)
with pd.ExcelWriter(
    practice_path, engine="openpyxl", mode="a", if_sheet_exists="replace"
) as writer:
    ranked.to_excel(writer, sheet_name="달성률순", index=False)
    style_table(
        writer.sheets["달성률순"], 1,
        formats={"B": MONEY, "C": MONEY, "D": PERCENT},
        widths={"A": 12, "B": 14, "C": 14, "D": 10},
    )
print(pd.ExcelFile(practice_path).sheet_names)
ranked

['요약', '달성률순']


,지점,금액,주간목표,달성률
4,광주점,4858200,4500000,1.079600
0,강남점,8004800,7500000,1.067307
3,부산점,7020900,7500000,0.936120
1,수원점,5975100,6500000,0.919246
2,대전점,4060700,5500000,0.738309
